# RQ2 · Complexity & construction tells

Two supporting questions:
1. **Complexity** — do paths get deeper / longer as readability falls?
2. **Construction** — *why*? Machine URLs correlate with query-construction
   characters and CMS/framework generation. We look for tells:
   Drupal (`/node/`), WordPress (`wp-`), ColdFusion (`.cfm`), classic ASP/.NET
   (`.asp`/`.aspx`), PHP (`.php`), and construction chars (`= ; + & %`).

These are **explanatory variables** for RQ1, not standalone findings.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root (this notebook lives in analysis/)
sys.path.insert(0, os.path.abspath('.'))
import config, readability as rb, eot_segments as es
import duckdb, pandas as pd, numpy as np
import matplotlib.pyplot as plt, seaborn as sns

sns.set_theme(style='whitegrid')
pd.set_option('display.max_rows', 120); pd.set_option('display.width', 200)
YEAR_ORDER = ['2004','2008','2012','2016','2020','2024']
CLS_COLORS = {'human':'#2c7fb8','acronym':'#fec44f','machine':'#de2d26'}

DBS = config.discover_domain_dbs('cdxj')
print(f"{len(DBS)}/15 domain DBs found:", ", ".join(DBS) or "(none — run on the server)")

## 1. Per-domain path features per year

One pass per domain over the deduplicated URL space (path incl. filename).

In [ ]:
FEATURE_SQL = r'''
WITH dedup AS (
  SELECT crawl_year, surtkey,
         min(regexp_extract(surtkey, '\)([^?]*)', 1)) AS p
  FROM eot_captures WHERE url NOT LIKE 'dns:%' GROUP BY 1,2)
SELECT crawl_year,
  count(*) AS unique_urls,
  median(CASE WHEN trim(p,'/')='' THEN 0 ELSE len(string_split(trim(p,'/'),'/')) END) AS median_depth,
  round(avg(length(p)),1) AS mean_path_len,
  round(100.0*avg(CASE WHEN regexp_matches(p,'[=;+&%]') THEN 1 ELSE 0 END),2) AS pct_construction,
  round(100.0*avg(CASE WHEN regexp_matches(lower(p),'(^|/)node/[0-9]') THEN 1 ELSE 0 END),2) AS pct_drupal,
  round(100.0*avg(CASE WHEN lower(p) LIKE '%wp-%' THEN 1 ELSE 0 END),2) AS pct_wordpress,
  round(100.0*avg(CASE WHEN lower(p) LIKE '%.cfm%' THEN 1 ELSE 0 END),2) AS pct_coldfusion,
  round(100.0*avg(CASE WHEN regexp_matches(lower(p),'\.aspx?($|[/?])') THEN 1 ELSE 0 END),2) AS pct_aspnet,
  round(100.0*avg(CASE WHEN lower(p) LIKE '%.php%' THEN 1 ELSE 0 END),2) AS pct_php,
  round(100.0*avg(CASE WHEN lower(p) LIKE '%.jsp%' THEN 1 ELSE 0 END),2) AS pct_jsp
FROM dedup GROUP BY 1 ORDER BY 1
'''
rows = []
for dom, path in DBS.items():
    con = duckdb.connect(str(path), read_only=True)
    df = con.sql(FEATURE_SQL).df(); con.close()
    df.insert(0,'domain',dom); rows.append(df)
feat = pd.concat(rows, ignore_index=True); feat['crawl_year']=feat['crawl_year'].astype(str)
feat.to_csv('rq2_path_features.csv', index=False)
feat

## 2. Complexity: median depth & mean path length over time

In [ ]:
fig, (a1,a2) = plt.subplots(1,2, figsize=(15,5))
for dom in DBS:
    d = feat[feat.domain==dom].sort_values('crawl_year')
    a1.plot(d.crawl_year, d.median_depth, 'o-', alpha=.5, label=dom)
    a2.plot(d.crawl_year, d.mean_path_len, 'o-', alpha=.5, label=dom)
a1.set_title('Median directory depth'); a1.set_xlabel('crawl year'); a1.set_ylabel('segments')
a2.set_title('Mean path length (chars)'); a2.set_xlabel('crawl year'); a2.set_ylabel('chars')
a1.legend(ncol=2, fontsize=7); plt.tight_layout()
plt.savefig('rq2_complexity.png', dpi=150, bbox_inches='tight'); plt.show()

## 3. Construction characters (`= ; + & %`) over time

A direct machine-assembly signal — the query-string era vs the clean-URL era.

In [ ]:
piv = feat.pivot(index='domain', columns='crawl_year', values='pct_construction')
piv = piv[[y for y in YEAR_ORDER if y in piv.columns]].reindex(
    [d for d in config.TARGET_DOMAINS if d in piv.index])
plt.figure(figsize=(9,7))
sns.heatmap(piv, annot=True, fmt='.1f', cmap='rocket_r',
            cbar_kws={'label':'% URLs with = ; + & %'}, linewidths=.5)
plt.title('Construction characters in URL paths'); plt.xlabel('crawl year'); plt.ylabel('')
plt.tight_layout(); plt.savefig('rq2_construction.png', dpi=150, bbox_inches='tight'); plt.show()

## 4. CMS / framework tells — the mechanism behind readability shifts

Pooled (URL-weighted) share of each tell over time. Rising Drupal/WordPress
often coincides with the clean-URL era; falling `.cfm`/`.asp` marks legacy
retirement. These help explain *why* RQ1 moves.

In [ ]:
tells = ['pct_drupal','pct_wordpress','pct_coldfusion','pct_aspnet','pct_php','pct_jsp']
w = feat.copy()
for t in tells: w[t+'_n'] = w[t]/100.0 * w['unique_urls']
g = w.groupby('crawl_year').agg(urls=('unique_urls','sum'),
        **{t:(t+'_n','sum') for t in tells})
for t in tells: g[t] = 100*g[t]/g['urls']
g = g.reindex([y for y in YEAR_ORDER if y in g.index])
ax = g[tells].plot(marker='o', figsize=(11,6))
ax.set_ylabel('% of unique URLs (federal, weighted)'); ax.set_xlabel('crawl year')
ax.set_title('CMS / framework tells over time')
ax.legend([t.replace('pct_','') for t in tells])
plt.tight_layout(); plt.savefig('rq2_tells.png', dpi=150, bbox_inches='tight'); plt.show()